# Colab → VSCode bridge

Run this notebook **on Colab** (T4 GPU runtime). It clones the stocker repo,
installs deps, then starts a Jupyter server tunneled via cloudflared.

Then on your **local machine**, open VSCode → Command Palette →
*Jupyter: Specify Jupyter Server for Connections* → paste the URL printed
by the last cell. Open `training/train_grpo.ipynb` locally; pick the remote
kernel; cells now execute on Colab's GPU while you edit in VSCode.

Caveat: the kernel runs on Colab, so file paths must match what the kernel
sees. The launcher `cd`s into `/content/stocker` and adds it to `sys.path`
before starting the server, so `import app...` works out of the box.
Files you save inside the notebook write to Colab's filesystem — push them
back via `git push` from a Colab cell, or mount Drive (cell 0).

In [ ]:
# OPTIONAL — mount Drive so artifacts under training/runs/ persist across sessions.
# Skip if you'll just download artifacts at the end of the run.
# from google.colab import drive
# drive.mount('/content/drive')
pass

In [ ]:
import os
REPO_URL = 'https://github.com/CRIMSONHydra/stocker.git'   # <-- edit me
WORKDIR  = '/content/stocker'
if not os.path.isdir(WORKDIR):
    assert '<your-username>' not in REPO_URL, 'Edit REPO_URL first.'
    !git clone {REPO_URL} {WORKDIR}
%cd {WORKDIR}
!git pull --rebase 2>/dev/null || true

In [ ]:
# Same dep set as train_grpo.ipynb's install cell.
!pip install -q -U 'transformers>=4.55' 'trl>=0.11' 'peft>=0.13' 'accelerate>=1.0' 'bitsandbytes>=0.43' 'datasets>=3.0'
!pip install -q --upgrade-strategy=only-if-needed yfinance mplfinance pyarrow 'pydantic>=2' pydantic-settings 'openai>=1' tensorboard 'huggingface_hub>=1.10' jupyter-server

In [ ]:
# Install cloudflared (free trycloudflare.com tunnel, no signup).
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

In [ ]:
# Pre-build the bundled dataset on Colab so VSCode-driven cells can
# `from app.data import loader` immediately. Idempotent.
!python scripts/build_dataset.py
!python scripts/validate_tasks.py

In [ ]:
import os, re, secrets, signal, socket, subprocess, sys, time, pathlib

TOKEN = secrets.token_urlsafe(24)
PORT  = 9988
LOG_J = pathlib.Path('/content/jupyter.log')
LOG_C = pathlib.Path('/content/cloudflared.log')
WORKDIR = '/content/stocker'

# We track the PIDs of OUR own children so re-running this cell can clean up
# without `pkill -f jupyter-server` (which would also kill Colab's backend
# and disconnect the runtime!).
PID_DIR = pathlib.Path('/tmp/stocker_launcher_pids'); PID_DIR.mkdir(exist_ok=True)

def _kill_prev(name: str) -> None:
    pf = PID_DIR / f'{name}.pid'
    if not pf.exists():
        return
    try:
        pid = int(pf.read_text().strip())
        os.kill(pid, signal.SIGKILL)
    except (ProcessLookupError, ValueError):
        pass
    pf.unlink(missing_ok=True)

_kill_prev('jupyter'); _kill_prev('cloudflared')
LOG_J.write_text(''); LOG_C.write_text('')

def _port_free(p: int) -> bool:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('127.0.0.1', p)) != 0

if not _port_free(PORT):
    raise RuntimeError(
        f'Port {PORT} still in use. Re-run after a moment, or restart the '
        f'runtime (Runtime > Restart session) and re-run all cells.'
    )

# --- jupyter-server -------------------------------------------------------
os.environ['PYTHONPATH'] = WORKDIR
j = subprocess.Popen(
    [
        sys.executable, '-m', 'jupyter', 'server',
        f'--ServerApp.token={TOKEN}',
        '--ServerApp.password=',
        f'--ServerApp.port={PORT}',
        '--ServerApp.ip=127.0.0.1',
        '--ServerApp.allow_origin=*',
        '--ServerApp.disable_check_xsrf=True',
        '--ServerApp.allow_remote_access=True',
        '--no-browser',
        f'--ServerApp.root_dir={WORKDIR}',
    ],
    stdout=open(LOG_J, 'w'),
    stderr=subprocess.STDOUT,
    cwd=WORKDIR,
)
(PID_DIR / 'jupyter.pid').write_text(str(j.pid))
time.sleep(3)
if j.poll() is not None:
    raise RuntimeError(f'jupyter-server died:\n{LOG_J.read_text()[-2000:]}')
print(f'jupyter-server up on 127.0.0.1:{PORT}  (pid={j.pid})')

# --- cloudflared (with --logfile, NOT shell redirection) ------------------
c = subprocess.Popen(
    ['cloudflared', 'tunnel',
     '--url', f'http://127.0.0.1:{PORT}',
     '--no-autoupdate',
     '--loglevel', 'info',
     '--logfile', str(LOG_C)],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
(PID_DIR / 'cloudflared.pid').write_text(str(c.pid))

print('cloudflared starting (typical: 5-30s) ', end='', flush=True)
url, deadline = None, time.time() + 120
URL_RE = re.compile(r'https://[a-z0-9-]+\.trycloudflare\.com')
while time.time() < deadline:
    if c.poll() is not None:
        print(f'\ncloudflared exited unexpectedly:\n{LOG_C.read_text()[-2000:]}', file=sys.stderr)
        break
    txt = LOG_C.read_text() if LOG_C.exists() else ''
    m = URL_RE.search(txt)
    if m:
        url = m.group(0)
        break
    print('.', end='', flush=True)
    time.sleep(2)
print()

if not url:
    raise RuntimeError(
        'cloudflared did not produce a URL within 120s.\n'
        '----- last 2KB of cloudflared log -----\n'
        + LOG_C.read_text()[-2000:]
        + '\n----- last 1KB of jupyter log -----\n'
        + LOG_J.read_text()[-1000:]
    )

FULL = f'{url}/?token={TOKEN}'
print('=' * 70)
print('Paste this into VSCode -> Jupyter: Specify Jupyter Server for Connections')
print()
print('   ', FULL)
print()
print('Smoke-test from anywhere:')
print(f'   curl -s "{FULL}/api/status" | head -c 200')
print('=' * 70)
print('\nKeep this Colab tab open — the tunnel dies when the runtime disconnects.')

## Connect from VSCode

1. **Install the Jupyter extension** in VSCode (Microsoft, ID `ms-toolsai.jupyter`) if you don't have it.
2. Open `training/train_grpo.ipynb` locally.
3. Command Palette → **`Jupyter: Specify Jupyter Server for Connections`** → *Existing* → paste the URL printed above.
4. In the kernel picker (top-right of the notebook), pick the remote runtime — VSCode shows it as `Python 3 (ipykernel)` under the trycloudflare host.
5. Run cells. Imports like `from app.council.llm import TransformersLLMClient` resolve because the kernel cwd is `/content/stocker`.

## Heartbeat

If you idle, Colab eventually evicts the runtime and the URL stops working.
Re-run the **`launch`** cell to get a new URL and re-specify the server in
VSCode. Long training runs should run inside a `nohup`-style guard or use
Colab Pro for the 24-hour runtime.

## Pulling artifacts back

Trained LoRAs and plots land in `/content/stocker/training/runs/<id>/`. To
get them onto your laptop:

```python
# in a Colab cell, after training
from google.colab import files
import shutil, glob
run = sorted(glob.glob('/content/stocker/training/runs/grpo_*'))[-1]
shutil.make_archive('/content/run', 'zip', run)
files.download('/content/run.zip')
```

Or `git add` + `git push` from a Colab cell to your fork.